# How much should you trust a cross-modal prediction?

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Ashford-A/UniVI/blob/main/docs/tutorials/experimental/prediction_uncertainty.ipynb)

> **Experimental.** `cross_modal_predict` returns one number per cell and feature. This notebook
> attaches uncertainty to those numbers and then checks, on held-out data, whether the uncertainty
> means anything.

Using RNA predicted from ATAC on 10x Multiome PBMCs:

1. **Posterior sampling.** Draw latent samples from each cell's ATAC posterior and decode them; the
   spread of the decoded values is the model's uncertainty about that cell's latent position.
2. **Does uncertainty track error?** Per cell and per gene, on held-out cells.
3. **Calibrated intervals.** Posterior spread ignores measurement noise, so raw intervals are too
   narrow. Split-conformal calibration on validation cells rescales them to a target coverage, which is
   then checked on test cells.
4. **Selective prediction.** If you keep only the most confident cells, how much better are the
   predictions (a risk–coverage curve)?
5. **An unseen cell type.** One cell type is removed from training entirely. Do its cells get higher
   uncertainty, and do the intervals still cover? Posterior spread is compared with a simple
   alternative novelty score, the distance to the nearest training cells in the latent space.

In [ ]:
import sys

if "google.colab" in sys.modules:
    %pip install -q "univi[tutorials]>=1.1" "pandas==2.2.3"

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import scipy.sparse as sp
import torch
from IPython.display import display
from scipy.stats import norm, spearmanr
from sklearn.metrics import roc_auc_score
from sklearn.neighbors import NearestNeighbors

import univi
import univi.datasets as uds
from univi import ModalityConfig, TrainingConfig, UniVIConfig, UniVIMultiModalVAE, UniVITrainer
from univi.evaluation import decode_from_latent, encode_adata
from univi.preprocessing import ATACPreprocessor, RNAPreprocessor, split_by_label
from univi.utils.seed import set_seed
from univi.workflows import make_loader

device = "cuda" if torch.cuda.is_available() else ("mps" if torch.backends.mps.is_available() else "cpu")
set_seed(0)
dense = lambda x: x.toarray() if sp.issparse(x) else np.asarray(x)
print(f"UniVI {univi.__version__} on {device}")

In [ ]:
N_EPOCHS = 400
BATCH_SIZE = 256
N_HVG = 2000
N_LSI = 101
N_SAMPLES = 30            # posterior samples per cell
COVERAGE = 0.90           # target coverage of the calibrated intervals
HELD_OUT_TYPE = None      # cell type removed from training; None picks one automatically (printed)

## 1. Data, with one cell type held out

All cells of `HELD_OUT_TYPE` are removed before splitting, so neither the preprocessing nor the model
sees them. By default the notebook picks the cell type whose size is the median among types with at
least 50 cells, which avoids both the dominant populations and tiny ones.

In [ ]:
data = uds.pbmc_multiome_10k()
rna, atac = data["rna"], data["atac"]
types = rna.obs["cell_type"].astype(str)
counts = types.value_counts()
if HELD_OUT_TYPE is None:
    eligible = counts[counts >= 50].sort_values()
    HELD_OUT_TYPE = eligible.index[len(eligible) // 2]
print(f"held-out cell type: {HELD_OUT_TYPE!r} ({counts[HELD_OUT_TYPE]} cells)")

seen = np.flatnonzero(types != HELD_OUT_TYPE)
split = split_by_label(types.iloc[seen], train_fraction=0.7, val_fraction=0.15, seed=0)
idx = {k: seen[v] for k, v in split.items()}
idx["unseen"] = np.flatnonzero(types == HELD_OUT_TYPE)
print({k: len(v) for k, v in idx.items()})

rna_prep = RNAPreprocessor(n_hvg=N_HVG, scale=True).fit(rna[idx["train"]])
atac_prep = ATACPreprocessor(n_components=N_LSI, drop_first=True, scale=True).fit(atac[idx["train"]])
parts = {k: {"rna": rna_prep.transform(rna[i]), "atac": atac_prep.transform(atac[i])} for k, i in idx.items()}

The validation set is larger than usual (15%) because it is used twice: for early stopping and, later,
for conformal calibration. Both uses only need cells the model was not trained on.

In [ ]:
cfg = UniVIConfig(
    latent_dim=30, beta=1.25, gamma=4.35, encoder_dropout=0.10, decoder_dropout=0.05,
    kl_anneal_start=50, kl_anneal_end=85, align_anneal_start=75, align_anneal_end=110,
    modalities=[ModalityConfig("rna", parts["train"]["rna"].n_vars, [512, 256, 128], [128, 256, 512]),
                ModalityConfig("atac", parts["train"]["atac"].n_vars, [128, 64], [64, 128])])
model = UniVIMultiModalVAE(cfg, loss_mode="v1", v1_recon="avg", normalize_v1_terms=True)
trainer = UniVITrainer(model, make_loader(parts["train"], batch_size=BATCH_SIZE, shuffle=True, drop_last=True),
                       make_loader(parts["val"], batch_size=1024),
                       TrainingConfig(n_epochs=N_EPOCHS, batch_size=BATCH_SIZE, lr=1e-3, weight_decay=1e-4,
                                      device=device, early_stopping=True, patience=50, best_epoch_warmup=110,
                                      log_every=50))
trainer.fit()
print("best epoch:", trainer.best_epoch)

## 2. Posterior samples of predicted RNA

`encode_adata(..., latent="modality_sample", random_state=s)` draws one sample from each cell's ATAC
posterior; decoding the samples with `decode_from_latent` gives `N_SAMPLES` predicted RNA profiles per
cell. Their mean is (up to sampling) the usual prediction; their standard deviation is the posterior
uncertainty.

In [ ]:
def predict_with_samples(split_name):
    draws = np.stack([decode_from_latent(model, encode_adata(model, parts[split_name]["atac"], modality="atac",
                                                             device=device, latent="modality_sample",
                                                             random_state=s), device=device)["rna"]
                      for s in range(N_SAMPLES)])
    return draws.mean(0), draws.std(0, ddof=1)


pred = {k: predict_with_samples(k) for k in ("val", "test", "unseen")}
obs = {k: dense(parts[k]["rna"].X) for k in pred}
print({k: v[0].shape for k, v in pred.items()})

## 3. Does the spread track the error?

Two checks on held-out test cells of seen types:

- **per cell**: Spearman correlation between a cell's mean predictive s.d. and its mean squared error;
- **per gene**: across cells, Spearman correlation between s.d. and absolute error, for every gene.

In [ ]:
mean_t, sd_t = pred["test"]
err_t = (obs["test"] - mean_t) ** 2
cell_rho = spearmanr(sd_t.mean(1), err_t.mean(1)).statistic
gene_rho = pd.Series([spearmanr(sd_t[:, j], np.sqrt(err_t[:, j])).statistic for j in range(sd_t.shape[1])],
                     index=parts["test"]["rna"].var_names)
print(f"per-cell Spearman(s.d., error): {cell_rho:.3f}")
print(f"per-gene Spearman(s.d., |error|): median {gene_rho.median():.3f}; "
      f"{(gene_rho > 0).mean():.1%} of genes positive")

fig, axes = plt.subplots(1, 2, figsize=(8.5, 3.2))
axes[0].scatter(sd_t.mean(1), err_t.mean(1), s=4, c="0.5")
axes[0].set(xlabel="mean predictive s.d. (cell)", ylabel="mean squared error (cell)", title="per cell")
axes[1].hist(gene_rho.dropna(), bins=40, color="0.6")
axes[1].axvline(0, c="k", lw=0.6)
axes[1].set(xlabel="Spearman(s.d., |error|) per gene", ylabel="genes", title="per gene")
plt.tight_layout()
plt.show()

## 4. Calibrated prediction intervals

The decoded samples describe uncertainty about the latent position, not measurement noise, so an
interval built directly from them (mean ± 1.645 s.d. for 90%) covers far fewer than 90% of observed
values. **Split-conformal prediction** fixes this with no distributional assumption: on validation
cells, compute the normalized residual `|observed − mean| / s.d.` and take, per gene, its quantile at
the target coverage (with the standard finite-sample correction). The interval `mean ± q × s.d.` then
covers at least the target fraction of values of a new cell, **provided** the new cell is exchangeable
with the validation cells (drawn from the same population). The unseen cell type breaks that
assumption, which Section 6 measures.

In [ ]:
eps = 1e-3 * np.median(pred["val"][1])
score_val = np.abs(obs["val"] - pred["val"][0]) / (pred["val"][1] + eps)
n_val = score_val.shape[0]
level = min(1.0, np.ceil((n_val + 1) * COVERAGE) / n_val)
q_gene = np.quantile(score_val, level, axis=0, method="higher")


def coverage(split_name, q):
    m, s = pred[split_name]
    inside = np.abs(obs[split_name] - m) <= q * (s + eps)
    return inside.mean(), np.median(2 * q * (s + eps))


z_naive = norm.ppf(0.5 + COVERAGE / 2)   # Gaussian quantile for a naive interval (1.645 for 90%)
rows = {}
for name in ("test", "unseen"):
    raw_cov, raw_width = coverage(name, np.full_like(q_gene, z_naive))
    cal_cov, cal_width = coverage(name, q_gene)
    rows[name] = {f"naive ±{z_naive:.3f} s.d. coverage": raw_cov, "conformal coverage": cal_cov,
                  "conformal median width": cal_width}
print(f"target coverage {COVERAGE:.0%}; per-gene conformal quantiles fit on {n_val} validation cells")
pd.DataFrame(rows).T.rename(index={"test": "held-out cells, seen types",
                                   "unseen": f"unseen type: {HELD_OUT_TYPE}"}).round(3)

Coverage near the target on held-out cells of seen types is what the conformal guarantee predicts (it
holds on average over genes and cells, not for every gene). A clear shortfall on the unseen type means
the intervals should not be trusted for populations absent from the reference.

## 5. Selective prediction: keep the confident cells

Rank held-out cells by their mean predictive s.d. and compute prediction quality (per-cell Pearson
correlation between predicted and observed profiles) for the most confident fraction. A curve that
rises as coverage shrinks means uncertainty can be used to filter predictions.

In [ ]:
def per_cell_r(o, m):
    o = o - o.mean(1, keepdims=True)
    m = m - m.mean(1, keepdims=True)
    return (o * m).sum(1) / (np.linalg.norm(o, axis=1) * np.linalg.norm(m, axis=1) + 1e-12)


r_cell = per_cell_r(obs["test"], mean_t)
order = np.argsort(sd_t.mean(1))
fractions = np.linspace(0.1, 1.0, 10)
curve = [r_cell[order[: max(1, int(f * len(order)))]].mean() for f in fractions]
rand = [np.mean([r_cell[np.random.default_rng(b).permutation(len(order))[: max(1, int(f * len(order)))]].mean()
                 for b in range(50)]) for f in fractions]
fig, ax = plt.subplots(figsize=(4.5, 3.2))
ax.plot(fractions, curve, marker="o", label="most confident first")
ax.plot(fractions, rand, ls="--", c="0.5", label="random order")
ax.set(xlabel="fraction of cells kept", ylabel="mean per-cell Pearson r", title="Risk-coverage")
ax.legend(frameon=False, fontsize=8)
plt.show()

## 6. Does uncertainty flag a cell type the model never saw?

Three candidate novelty scores for each cell, compared by how well they separate unseen-type cells from
held-out cells of seen types (AUROC; 0.5 = no separation):

- **predictive s.d.**: mean s.d. of the decoded samples;
- **posterior variance**: the ATAC encoder's own variance, summed over latent dimensions;
- **latent kNN distance**: mean distance to the 15 nearest training cells in the ATAC latent space.

Variational autoencoders are not guaranteed to be more uncertain on new inputs, so any of these can
fail; the comparison shows which is informative here.

In [ ]:
def posterior_variance(adata):
    x = torch.as_tensor(dense(adata.X), dtype=torch.float32, device=device)
    model.eval()
    with torch.no_grad():
        _, logvar = model.encode_modalities({"atac": x})
    return logvar["atac"].exp().sum(1).cpu().numpy()


z_train = encode_adata(model, parts["train"]["atac"], modality="atac", device=device, latent="modality_mean")
nn = NearestNeighbors(n_neighbors=15).fit(z_train)
knn_dist = {k: nn.kneighbors(encode_adata(model, parts[k]["atac"], modality="atac", device=device,
                                          latent="modality_mean"))[0].mean(1) for k in ("test", "unseen")}
scores = {"predictive s.d.": {k: pred[k][1].mean(1) for k in ("test", "unseen")},
          "posterior variance": {k: posterior_variance(parts[k]["atac"]) for k in ("test", "unseen")},
          "latent kNN distance": knn_dist}
y = np.r_[np.zeros(len(idx["test"])), np.ones(len(idx["unseen"]))]
auroc = pd.Series({name: roc_auc_score(y, np.r_[s["test"], s["unseen"]]) for name, s in scores.items()},
                  name=f"AUROC, {HELD_OUT_TYPE} vs seen types")
display(auroc.round(3).to_frame())

fig, axes = plt.subplots(1, len(scores), figsize=(3.3 * len(scores), 2.8))
for ax, (name, s) in zip(axes, scores.items()):
    bins = np.histogram_bin_edges(np.r_[s["test"], s["unseen"]], 40)
    ax.hist(s["test"], bins=bins, alpha=0.6, density=True, label="seen types")
    ax.hist(s["unseen"], bins=bins, alpha=0.6, density=True, label=HELD_OUT_TYPE)
    ax.set_title(f"{name}\nAUROC {auroc[name]:.2f}", fontsize=9)
axes[0].legend(frameon=False, fontsize=7)
plt.tight_layout()
plt.show()

## Using this in practice

- Report predicted features **with** their conformal intervals, and state the calibration population.
  The coverage guarantee holds for new cells drawn from the same population as the calibration cells.
- Use a novelty score (whichever separated best in Section 6) to flag query cells that look unlike the
  reference, and treat their predictions as unreliable regardless of interval width.
- Everything here works for any source and target modality: replace `"atac"` and `"rna"` in
  `predict_with_samples`, for example to put intervals on protein predicted from RNA in CITE-seq.